In [112]:
# pip install python-calamine

In [113]:
import pdb
from datetime import datetime
from datetime import timedelta
import pandas as pd
import numpy as np 
# from pandas.io.json import json_normalize
from pandas import json_normalize
import json
import psycopg2
from datetime import datetime
from dateutil.relativedelta import relativedelta
from urllib.parse import quote
from sqlalchemy.engine import create_engine
import warnings
warnings.filterwarnings("ignore") 

In [114]:
# sas_config = r"C:\Users\mononita.mitra\OneDrive - UGRO Capital\Desktop\Mononita Data 14-09-2023\Mononita\GroMicro Login Analysis\config_ods.json"
# sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods 2.json"
sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods_Saurabh.json"
def connection_from_sas():
    try:
        f = open(sas_config)
        data = json.load(f)

 

        database = data.get("database")
        user = data.get("user")
        password = data.get("password")
        host = data.get("host")
        port = data.get("port")

 

        connection = psycopg2.connect(database=database, user=user, password=password, host=host, port=port)
        return connection

 

    except Exception as error:
        print(error)

In [115]:
connection = connection_from_sas()

In [116]:
# SCF details
# sql_query_1 =  """select 
# object_pri_key_1 
# from tb_udhyam_details"""

sql_query_1 = """
select  Laf_id, 
cast (datetime_login as date) as Login_date,
gender, unique_identity_number as PAN from Ckyc a
left join laf_detail_status b on b.laf_number=a.laf_id
where gender in ('female','Female','FeMale','FEMale','FEMALE')
"""

# select  a.Laf_id, jcd.agreementno as lan,
# cast (datetime_login as date) as Login_date,
#  string_agg(gender,', ') as gender, string_agg(a.type_of_borrower ,', ') as type_of_borrower,string_agg(unique_identity_number,', ') as pan  from Ckyc a
# left join laf_detail_status b on b.laf_number=a.laf_id
# left join jayam_contract_details jcd on jcd.laf_id =b.laf_number 
# where gender <>''--- and jcd.agreementno ='UGBANMC0000101334'
# group by a.Laf_id, jcd.agreementno ,
# datetime_login



# ods.msl_loan_detail for gender

In [117]:
# jayam details
sql_query_2 = """
select  
object_pri_key_1,
agreementno,
lds.pan_number, 
activitydescription,
a.industry,
udyamregistrationno,
subsector,
organizationtype,
enterprisetype,
majoractivity from tb_udhyam_details a
left join jayam_contract_details b on a.object_pri_key_1=b.laf_id
left join laf_detail_status lds on lds.laf_number  = a.object_pri_key_1 

"""

In [118]:
# # qualtech details
# sql_query_3 = """
# select  
# QLD.prospectcode as lan_number,
# QLD.prospectid as agreementid,
# QLS.loanstatusname,
# '' as gstin,
# coalesce(QAM.fname,'')||' '||coalesce(QAM.lname,'') as customer_name,
# QAM.panno as PAN,
# '' as gst_address,
# coalesce(QAD.flatno,'')||' '||coalesce(QAD.floorno,'')||' '||coalesce(QAD.buildingname,'') as business_address,
# QSM.statemastername as state_of_customer,
# '' as branch
# from qualtech_qt_applicantmaster QAM
# inner join qualtech_qt_applicantaddress QAD
# on QAM.applicantid = QAD.applicantid
# left join qualtech_qm_statemaster QSM
# on QAD.state = QSM.statemasterid
# left join qualtech_qt_loandetail QLD
# on QAM.applicantid = QLD.customerid
# left join qualtech_qm_loan_status QLS
# on QLD.loansubstatusid = substr(QLS.loanstatusid,1,10)
# where QAD.addresstype = '1000000014'
# """

In [119]:
df = pd.read_sql_query(sql_query_1, connection)

In [120]:
df=df.drop_duplicates('laf_id')

In [121]:
df.shape

(418469, 4)

In [122]:
df2 = pd.read_sql_query(sql_query_2, connection)

In [123]:
df2.columns

Index(['object_pri_key_1', 'agreementno', 'pan_number', 'activitydescription',
       'industry', 'udyamregistrationno', 'subsector', 'organizationtype',
       'enterprisetype', 'majoractivity'],
      dtype='object')

In [124]:
df2['enterprisetype'] = np.where(df2['enterprisetype'].str.contains('MICRO | Micro', na=False), 'Micro', df2['enterprisetype'])

In [125]:
df2['enterprisetype'] = np.where(df2['enterprisetype'].str.contains('SMALL | Small', na=False), 'Small', df2['enterprisetype'])

In [126]:
df2['enterprisetype'] = np.where(df2['enterprisetype'].str.contains('MEDIUM | Medium', na=False), 'Medium', df2['enterprisetype'])

In [127]:
df2['majoractivity'] = np.where(df2['majoractivity'].str.contains('TRADING | Trading', na=False), 'Trading', df2['majoractivity'])

In [128]:
df2['majoractivity'].unique()

array(['TRADING', 'Services', 'Trading', 'Manufacturing'], dtype=object)

In [129]:
df2['Source'] = np.where(df2['object_pri_key_1'].notnull(), 'tb_udhyam','NA')

In [130]:
# df.rename(columns={'old_name': 'new_name'}, inplace=True)


In [131]:
sql_query_3 = """
 select * from msme_classification
 """

In [132]:
df3 = pd.read_sql_query(sql_query_3, connection)

In [133]:
df3["PQR_LANS"] = 'MSL' + df3["loandetail_lenderloanacctid"]

In [134]:
# df.rename(columns={'old_name': 'new_name'}, inplace=True)

df3.rename(columns={'nic_2_digit': 'activitydescription'}, inplace=True)
df3.rename(columns={'PQR_LANS': 'agreementno'}, inplace=True)
df3.rename(columns={'ld_ui_userid': 'object_pri_key_1'}, inplace=True)
df3.rename(columns={'industry': 'majoractivity'}, inplace=True)
df3.rename(columns={'sub_industry': 'industry'}, inplace=True)
df3.rename(columns={'ld_ui_profiledata_workudyamnumber': 'udyamregistrationno'}, inplace=True)
df3.rename(columns={'industryclassification_industrygroupname': 'subsector'}, inplace=True)

In [135]:
# df3['subsector'] = np.where(df3['udyamregistrationno'].notnull(), df3['datana'],'NA')

In [136]:
df3.columns

Index(['loandetail_lenderloanacctid', 'object_pri_key_1',
       'udyamaadhaardetails_majoractivity', 'udyamregistrationno', 'length',
       'subsector', 'pan_number', 'customer_name', 'year', 'month', 'job_run',
       'bsrcode', 'enterprisetype', 'majoractivity', 'activitydescription',
       'udyamaadhaardetails_organizationtype', 'industry', 'agreementno'],
      dtype='object')

In [137]:
df3['Source'] = np.where(df3['agreementno'].notnull(), 'MSL','NA')

In [138]:
df3.columns

Index(['loandetail_lenderloanacctid', 'object_pri_key_1',
       'udyamaadhaardetails_majoractivity', 'udyamregistrationno', 'length',
       'subsector', 'pan_number', 'customer_name', 'year', 'month', 'job_run',
       'bsrcode', 'enterprisetype', 'majoractivity', 'activitydescription',
       'udyamaadhaardetails_organizationtype', 'industry', 'agreementno',
       'Source'],
      dtype='object')

In [139]:
columns_to_keep = ['object_pri_key_1','udyamregistrationno','majoractivity','activitydescription','industry','agreementno','pan_number','enterprisetype','subsector','Source']

df4 = df3[columns_to_keep]

In [140]:
df_result = pd.concat([df2, df4], ignore_index=True, sort=False)
# DataFrame.append
# Series.append
# df_result = df2.append(df4, ignore_index=True)

In [141]:
df_result['majoractivity'].unique()

array(['TRADING', 'Services', 'Trading', 'Manufacturing', None, ''],
      dtype=object)

In [142]:
df_result['Source'].unique()

array(['tb_udhyam', 'MSL'], dtype=object)

In [143]:
df_result.rename(columns={'activitydescription': 'activitydesc'}, inplace=True)

In [144]:
df_result['activitydescription'] = df_result['activitydesc'].astype(str) + ', ' + df_result['industry'].astype(str) + ', ' + df_result['subsector'].astype(str)

In [145]:
df_result['activitydescription'] = df_result['activitydescription'].str.lower()

In [146]:
df_result['Ind_Group'] = np.where(df_result['object_pri_key_1'].notnull(), '','')

In [147]:
df_perfios = pd.read_excel(r"D:\Saurabh\Reports\RBI Reporting\perfios ops all udhyam industry classification.xlsx","perfios") # reads the first sheet of your excel file

In [148]:
df_perfios['object_pri_key_1'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')
df_perfios['agreementno'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')
df_perfios['subsector'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')
df_perfios['industry'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')
df_perfios['Ind_Group'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')
df_perfios['activitydesc'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')

df_perfios['majoractivity'] = np.where(df_perfios['activitydescription'].notnull(), 'NA','NA')

In [149]:
df_perfios['Source'] = np.where(df_perfios['activitydescription'].notnull(), 'perfios','NA')

In [150]:
df_perfios.columns

Index(['pan_number', 'recevied in', 'Case Id', 'majoractivity',
       'enterprisetype', 'activitydescription', 'udyamregistrationno',
       'object_pri_key_1', 'agreementno', 'subsector', 'industry', 'Ind_Group',
       'activitydesc', 'Source'],
      dtype='object')

In [151]:
columns_to_keep = ['object_pri_key_1','agreementno','activitydesc','Ind_Group','udyamregistrationno','majoractivity','activitydescription','industry','pan_number','enterprisetype','subsector','Source']

df_perfios_1 = df_perfios[columns_to_keep]

In [152]:
# df_perfios_1.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df_perfios_1.xlsx", index=False)

In [153]:
df_result = pd.concat([df_result, df_perfios_1], ignore_index=True, sort=False)

In [154]:
df_ops = pd.read_excel(r"D:\Saurabh\Reports\RBI Reporting\perfios ops all udhyam industry classification.xlsx","ops") # reads the first sheet of your excel file

In [155]:
df_ops["pqr_lans"] = 'MSL' + df_ops["agreementno"].astype(str)

In [156]:
# df_ops = df_ops.drop(columns=['pqr_lans1'])

In [157]:
df_ops['object_pri_key_1'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')
df_ops['pan_number'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')
df_ops['activitydesc'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')
df_ops['industry'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')
df_ops['subsector'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')
df_ops['Ind_Group'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')

df_ops['majoractivity'] = np.where(df_ops['activitydescription'].notnull(), 'NA','NA')

In [158]:
df_ops['Source'] = np.where(df_ops['activitydescription'].notnull(), 'ops','NA')

In [159]:
df_ops

,agreementno,majoractivity,activitydescription,enterprisetype,udyamregistrationno,Source,PAN,pqr_lans,object_pri_key_1,pan_number,activitydesc,industry,subsector,Ind_Group
0,48271,NA,"Wholesale trade, except of motor vehicles and...",Micro,UDYAM-WB-09-0008284,ops,NaN,MSL48271,NA,NA,NA,NA,NA,NA
1,48307,NA,Telecommunications,Micro,UDYAM-BR-03-0005261,ops,NaN,MSL48307,NA,NA,NA,NA,NA,NA
2,48479,NA,Telecommunications,Micro,UDYAM-UP-17-0028418,ops,NaN,MSL48479,NA,NA,NA,NA,NA,NA
3,48509,NA,Repair of computers and personal and househol...,Micro,UDYAM-TN-06-0005110,ops,NaN,MSL48509,NA,NA,NA,NA,NA,NA
4,48542,NA,"Financial service activities, except insuranc...",Micro,UDYAM-KR-02-0048563,ops,NaN,MSL48542,NA,NA,NA,NA,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478239,822080,NA,Service and Utilities,Micro,NaN,ops,FTVPA0548G,MSL822080,NA,NA,NA,NA,NA,NA
478240,827093,NA,Retail and Shopping,Micro,NaN,ops,CUPPD2242C,MSL827093,NA,NA,NA,NA,NA,NA
478241,827491,NA,Service and Utilities,Micro,NaN,ops,AMVPK3582A,MSL827491,NA,NA,NA,NA,NA,NA
478242,827877,NA,Retail and Shopping,Micro,NaN,ops,CKSPS8918P,MSL827877,NA,NA,NA,NA,NA,NA


In [160]:
df_result = pd.concat([df_result, df_ops], ignore_index=True, sort=False)

In [161]:
df_result['Ind_Group'] = np.where(df_result['object_pri_key_1'].notnull(), 'NA','NA')

In [162]:
df_result['activitydescription'] = df_result['activitydescription'].str.lower()

In [163]:
df_result.loc[
    df_result['Ind_Group'].isin(['NA']) &
           df_result['activitydescription'].str.contains('restaurant')  , 
        'Ind_Group']="restaurant"

In [164]:
# df_result['activitydescription'] = df_result['activitydescription'].str.strip()

In [165]:
import re
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('food', case=False)|df_result['activitydescription'].str.contains('chocolate')  | df_result['activitydescription'].str.contains('blending') | df_result['activitydescription'].str.contains('biscuit')  |df_result['activitydescription'].str.contains('sauces')  |df_result['activitydescription'].str.contains('cafeterias') 
| df_result['activitydescription'].str.contains('dairy')| df_result['activitydescription'].str.contains('milk')| df_result['activitydescription'].str.contains('cream')| df_result['activitydescription'].str.contains('butter')
| df_result['activitydescription'].str.contains('fruit')| df_result['activitydescription'].str.contains('bakery')| df_result['activitydescription'].str.contains('vegetable')| df_result['activitydescription'].str.contains('coffee')
| df_result['activitydescription'].str.contains('seed')| df_result['activitydescription'].str.contains('drinks')| df_result['activitydescription'].str.contains('cafe')| df_result['activitydescription'].str.contains('fast-food')), 
        'Ind_Group']="Food"

In [166]:
columns_to_keep = ['agreementno','activitydescription','Ind_Group']

df_result_test = df_result[columns_to_keep]

In [167]:
# filtered_df = currrent_rbi_file_2[currrent_rbi_file_2['LAN_Number'] != 'HCFAHMMLN00001005844']
df_result_test = df_result_test[(df_result_test["agreementno"]=="HCFAJMSEC00001012127")] #Filtering dataframe
df_result_test

,agreementno,activitydescription,Ind_Group
10938,HCFAJMSEC00001012127,"flour milling, manufacture of food products, m...",Food
16331,HCFAJMSEC00001012127,"other semi-processed, processed or instant foo...",Food
181194,HCFAJMSEC00001012127,"other semi-processed, processed or instant foo...",Food
325437,HCFAJMSEC00001012127,"flour milling, manufacture of food products, m...",Food


In [168]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('agriculture'))  , 
        'Ind_Group']="agriculture"

In [169]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('mining')  | df_result['activitydescription'].str.contains('coal')), 
        'Ind_Group']="Mining"

In [170]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('textiles')  | df_result['activitydescription'].str.contains('clothing')| df_result['activitydescription'].str.contains('fabrics')| df_result['activitydescription'].str.contains('textile')
| df_result['activitydescription'].str.contains('garments')| df_result['activitydescription'].str.contains('wearing')), 
        'Ind_Group']="Textiles"

In [171]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('leather')  | df_result['activitydescription'].str.contains('bags')| df_result['activitydescription'].str.contains('footwear')
| df_result['activitydescription'].str.contains('sport')), 
        'Ind_Group']="Leather"

In [172]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('wood')  | df_result['activitydescription'].str.contains('furniture')| df_result['activitydescription'].str.contains('plywood')| df_result['activitydescription'].str.contains('door')), 
        'Ind_Group']="wood"

In [173]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('tobacco')  | df_result['activitydescription'].str.contains('beverage')), 
        'Ind_Group']="Beverage & Tobacco"

In [174]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('paper')  | df_result['activitydescription'].str.contains('box')| df_result['activitydescription'].str.contains('books')| df_result['activitydescription'].str.contains('print')), 
        'Ind_Group']="paper"

In [175]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('petrol')  | df_result['activitydescription'].str.contains('coal')| df_result['activitydescription'].str.contains('crude')
| df_result['activitydescription'].str.contains('petroleum')), 
        'Ind_Group']="Petroleum, Coal Products & Nuclear Fuels"

In [176]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('chemicals')  | df_result['activitydescription'].str.contains('perfumes')| df_result['activitydescription'].str.contains('pharma')), 
        'Ind_Group']="Chemicals"

In [177]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('rubber')  | df_result['activitydescription'].str.contains('tyres')), 
        'Ind_Group']="Rubber"

In [178]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('glass')) , 
        'Ind_Group']="Glass"

In [179]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('cement')) , 
        'Ind_Group']="Cement"

In [180]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('estate') | df_result['activitydescription'].str.contains('construction')), 
        'Ind_Group']="Construction"

In [181]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('machinery')| df_result['activitydescription'].str.contains('equipment')|df_result['activitydescription'].str.contains('metal')| df_result['activitydescription'].str.contains('iron')| df_result['activitydescription'].str.contains('zinc')) , 
        'Ind_Group']="metal"

In [182]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('engineering')| df_result['activitydescription'].str.contains('machinery')| df_result['activitydescription'].str.contains('equipment')| df_result['activitydescription'].str.contains('engineering')) , 
        'Ind_Group']="Engineering"

In [183]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('power')| df_result['activitydescription'].str.contains('solar')| df_result['activitydescription'].str.contains('generator')
| df_result['activitydescription'].str.contains('electri')), 
        'Ind_Group']="Power"

In [184]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           df_result['activitydescription'].str.contains('motor')| df_result['activitydescription'].str.contains('vehical')| df_result['activitydescription'].str.contains('machine'), 
        'Ind_Group']="vehical"

In [185]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('road')| df_result['activitydescription'].str.contains('road')| df_result['activitydescription'].str.contains('road')), 
        'Ind_Group']="road"

In [186]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('freight')| df_result['activitydescription'].str.contains('freight')| df_result['activitydescription'].str.contains('cargo')|df_result['activitydescription'].str.contains('logistic')
| df_result['activitydescription'].str.contains('courier')), 
        'Ind_Group']="Logistical"

In [187]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('hotel')| df_result['activitydescription'].str.contains('restaurant')| df_result['activitydescription'].str.contains('hostels')
| df_result['activitydescription'].str.contains('accommodation')), 
        'Ind_Group']="Hotel" 

In [188]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('transport')| df_result['activitydescription'].str.contains('travel')), 
        'Ind_Group']="Transport"

In [189]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('school')| df_result['activitydescription'].str.contains('education')| df_result['activitydescription'].str.contains('educational')), 
        'Ind_Group']="Education"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('admin')| df_result['activitydescription'].str.contains('facilities')| df_result['activitydescription'].str.contains('social')
| df_result['activitydescription'].str.contains('advertising')| df_result['activitydescription'].str.contains('maintenance')| df_result['activitydescription'].str.contains('security')
| df_result['activitydescription'].str.contains('service')| df_result['activitydescription'].str.contains('services')|df_result['activitydescription'].str.contains('consultancy')| df_result['activitydescription'].str.contains('care')
| df_result['activitydescription'].str.contains('profession')| df_result['activitydescription'].str.contains('agencies')| df_result['activitydescription'].str.contains('agency')| df_result['activitydescription'].str.contains('television')
| df_result['activitydescription'].str.contains('financial')| df_result['activitydescription'].str.contains('rent')
| df_result['activitydescription'].str.contains('music')| df_result['activitydescription'].str.contains('activities')), 
        'Ind_Group']="Professional Services"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('medical')| df_result['activitydescription'].str.contains('hospital')| df_result['activitydescription'].str.contains('health')
| df_result['activitydescription'].str.contains('veterinary')), 
        'Ind_Group']="Health"  

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('plastic')|df_result['activitydescription'].str.contains('plastics')), 
        'Ind_Group']="Plastic"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           df_result['activitydescription'].str.contains('telecom'), 
        'Ind_Group']="Telecom"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('computer')), 
        'Ind_Group']="Computer"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('software')), 
        'Ind_Group']="software"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('tobacco')|df_result['activitydescription'].str.contains('beverage')), 
        'Ind_Group']="Beverage & Tobacco"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('water')), 
        'Ind_Group']="Water Treatment Plants"

In [ ]:
# df_result.loc[
#     df_result['Ind_Group'].isin(['NA']) &
#            df_result['activitydescription'].str.contains('articles'), 
#         'Ind_Group']="2 s. Other Industries"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('retail')|df_result['activitydescription'].str.contains('grocery')), 
        'Ind_Group']="Retail"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('wholesale')), 
        'Ind_Group']="Wholesale"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('waste')|df_result['activitydescription'].str.contains('sewerage')), 
        'Ind_Group']="Solid Waste Management"

In [ ]:
# filtered_df = currrent_rbi_file_2[currrent_rbi_file_2['LAN_Number'] != 'HCFAHMMLN00001005844']
df_result_test = df_result[(df_result["agreementno"]=="HCFAJMSEC00001012127")] #Filtering dataframe
df_result_test

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('articles') | df_result['activitydescription'].str.contains('manufacturing')|df_result['activitydescription'].str.contains('work')|df_result['activitydescription'].str.contains('manufacture')) , 
        'Ind_Group']="2.0_Others"


# df_result.loc[
#     (df_result['Ind_Group'].isin(['NA'])) &
#     (df_result['activitydescription'].str.contains('articles') |
#      df_result['activitydescription'].str.contains('manufacturing')|
#      df_result['activitydescription'].str.contains('work')|
#      df_result['activitydescription'].str.contains('manufacture')
#      ) , 'Ind_Group'] = "2.0_Others"
 

In [ ]:
columns_to_keep = ['agreementno','activitydesc','Ind_Group']

df_result_test1 = df_result[columns_to_keep]

In [ ]:
# filtered_df = currrent_rbi_file_2[currrent_rbi_file_2['LAN_Number'] != 'HCFAHMMLN00001005844']
df_result_test2 = df_result_test1[(df_result_test1["agreementno"]=="HCFAJMSEC00001012127")] #Filtering dataframe
df_result_test2

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('none,') |df_result['activitydescription'].str.contains('garden')|df_result['activitydescription'].str.contains('toys')
|df_result['activitydescription'].str.contains('toys')), 
        'Ind_Group']="3.10_Others"

In [ ]:
df_result.loc[
    (df_result['Ind_Group'].isin(['NA'])) &
           (df_result['activitydescription'].str.contains('electronic,')),
        'Ind_Group']="electronics"

In [ ]:
df_result_pivot = df_result.pivot_table(values=['object_pri_key_1'], index=['Ind_Group'],
#                       df8 = df7.pivot_table(values=['laf','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'], columns=['DPD_bucket'],
                        aggfunc="count", fill_value=0, margins=True,margins_name= 'Total')
#                         aggfunc="sum", fill_value=0)
df_result_pivot

In [ ]:
df_result_blanks = df_result[(df_result["Ind_Group"]=="") & (df_result["activitydescription"]!="")] #Filtering dataframe

In [ ]:
df_result['Ind_Group'].unique()

In [ ]:
# chunk_size = 1000000
# with pd.ExcelWriter(r"C:\Users\saurabh.khopkar\Downloads\df_result.xlsx", engine='openpyxl') as writer:
#     for i, start_row in enumerate(range(0, len(df_result), chunk_size)):
#         df_chunk = df_result.iloc[start_row:start_row + chunk_size]
#         sheet_name = f'Sheet_{i+1}'
#         df_chunk.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
# chunk_size1 > 1000002
# with pd.ExcelWriter(r"C:\Users\saurabh.khopkar\Downloads\df_result_1.xlsx", engine='openpyxl') as writer:
#     for i, start_row in enumerate(range(0, len(df_result), chunk_size1)):
#         df_chunk1 = df_result.iloc[start_row:start_row + chunk_size1]
#         sheet_name = f'Sheet_{i+1}'
#         df_chunk1.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
# # df_result = df_result.to_excel(r"C:\Users\saurabh.khopkar\Downloads\CKYC.xlsx", chunksize=chunk_size)
# df_result.to_excel(r"C:\Users\saurabh.khopkar\Downloads\CKYC.xlsx", index=False, chunksize=chunk_size)

In [ ]:
# file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
# one_day_ago = datetime.now() - timedelta(days=0)
# cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
# filename="df_result" + str(cur_time) + ".xlsx"
# df_result.to_excel(file_path + filename, index=False)

In [ ]:
# df3 = pd.read_sql_query(sql_query_3, connection)

In [ ]:
# df
# currentDateTime = datetime.now().strftime("%m-%d-%Y %H:%M:%S %p")
# df.to_excel(r"C:\Users\saurabh.khopkar\Downloads\CKYC.xlsx", index=False)

In [ ]:
# from datetime import datetime as dt
# mask = '%d%m%Y'
# dte = dt.now().strftime(mask)
# fname = r"C:\Users\saurabh.khopkar\Downloads\CKYC_As_On_{}.xlsx".format(dte)
# df.to_excel(fname)

In [ ]:
# file_path = 'C:\\Users\\saurabh.khopkar\\Downloads'
# filename="CKYC as on"
# date = datetime.now().strftime("%d-%m-%Y_%I:%M_%p")
# df.to_excel(f"{file_path}{filename}{date}.xlsx")

In [ ]:
columns_to_keep = ['object_pri_key_1','agreementno','pan_number','udyamregistrationno','enterprisetype','majoractivity','activitydescription','Source','Ind_Group']

In [ ]:
df_ind_fnl = df_result[columns_to_keep]

In [ ]:
df_ind_fnl['Ind_Monthly'] = np.where(df_ind_fnl['object_pri_key_1'].notnull(), 'NA','NA')

In [ ]:
df_ind_fnl['Ind_Group'] = df_ind_fnl['Ind_Group'].str.lower()

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('food')  | df_ind_fnl['Ind_Group'].str.contains('beverage & tobacco')), 
        'Ind_Monthly']="2.a Food Processing"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('textiles')), 
        'Ind_Monthly']="2.b Textiles"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('chemicals')), 
        'Ind_Monthly']="2.d Chemicals & Chemical Products"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('metal')), 
        'Ind_Monthly']="2.e Basic Metal & Metal Product"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('metal')), 
        'Ind_Monthly']="2.e Basic Metal & Metal Product"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('computer')), 
        'Ind_Monthly']="2.f.2 Computer Hardware"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('computer')), 
        'Ind_Monthly']="2.f.2 Computer Hardware"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('vehical')), 
        'Ind_Monthly']="2.g Vehicle Parts & Transport Equipment"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('mining')), 
        'Ind_Monthly']="2.h.1.4 Thermal Power Generation"

In [ ]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
         (  df_ind_fnl['Ind_Group'].str.contains('power')), 
        'Ind_Monthly']="2.h.1.5 Other Power"

In [ ]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA']))&
           (df_ind_fnl['Ind_Group'].str.contains('solid waste management')| df_ind_fnl['Ind_Group'].str.contains('water treatment plants')), 
        'Ind_Monthly']="2.h.8 Other Infrastructure, please specify"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('construction') |df_ind_fnl['Ind_Group'].str.contains('cement')|df_ind_fnl['Ind_Group'].str.contains('engineering')), 
        'Ind_Monthly']="2.i Construction & Civil Engineering"

In [ ]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('plastic')| df_ind_fnl['Ind_Group'].str.contains('paper')), 
        'Ind_Monthly']="2.j Packaging Activities"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('electronics')), 
        'Ind_Monthly']="2.f.3 Other Electronics"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('wood')| df_ind_fnl['Ind_Group'].str.contains('leather')| df_ind_fnl['Ind_Group'].str.contains('glass')), 
        'Ind_Monthly']="2.k Other Industries, please specify"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('wholesale')), 
        'Ind_Monthly']="3.6.1 Wholesale Trade"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('retail')), 
        'Ind_Monthly']="3.6.2 Retail Trade"

In [ ]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('software')), 
        'Ind_Monthly']="3.b IT related Services including Computer Software"

In [ ]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('education')), 
        'Ind_Monthly']="3.c Educational Institutions"

In [ ]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('health')), 
        'Ind_Monthly']="3.d Health related Services"

In [243]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA']) )&
           (df_ind_fnl['Ind_Group'].str.contains('hotel') | df_ind_fnl['Ind_Group'].str.contains('restaurant')), 
        'Ind_Monthly']="3.e Tourism, Hotel & Restaurants"  

In [244]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('professional services') ), 
        'Ind_Monthly']="3.f Professional Services"  

In [245]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('construction')) , 
        'Ind_Monthly']="3.h Commercial Real Estate"  

In [246]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('3.10_others')|df_ind_fnl['Ind_Group'].str.contains('3.10_others')) , 
        'Ind_Monthly']="3.k Other Services, please specify"  

In [247]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('agriculture')) , 
        'Ind_Monthly']="agriculture"  

In [248]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('petroleum')|df_ind_fnl['Ind_Group'].str.contains('coal')|df_ind_fnl['Ind_Group'].str.contains('nuclear')|df_ind_fnl['Ind_Group'].str.contains('fuel')), 
        'Ind_Monthly']="2.c Petroleum, Coal Products & Nuclear Fuels"

In [249]:
df_ind_fnl.loc[
   ( df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('road')),
        'Ind_Monthly']="2.h.3 Roads"

In [250]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('road')),
        'Ind_Monthly']="2.h.3 Roads"

In [251]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
          ( df_ind_fnl['Ind_Group'].str.contains('rubber')),
        'Ind_Monthly']="2.j Packaging Activities"

In [252]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('telecom')),
        'Ind_Monthly']="3.f Professional Services"

In [253]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('transport')),
        'Ind_Monthly']="3.a Transport Operators"

In [254]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('transport') | df_ind_fnl['Ind_Group'].str.contains('logistical')),
        'Ind_Monthly']="3.a Transport Operators"

In [255]:
df_ind_fnl.loc[
    (df_ind_fnl['Ind_Monthly'].isin(['NA'])) &
           (df_ind_fnl['Ind_Group'].str.contains('2.0_Others')|df_ind_fnl['Ind_Group'].str.contains('2.0_others')),
        'Ind_Monthly']="2.k Other Industries, please specify"

In [256]:
# filtered_df = currrent_rbi_file_2[currrent_rbi_file_2['LAN_Number'] != 'HCFAHMMLN00001005844']
df_ind_fnl_test = df_ind_fnl[(df_ind_fnl["agreementno"]=="HCFAJMSEC00001012127")] #Filtering dataframe
df_ind_fnl_test

,object_pri_key_1,agreementno,pan_number,udyamregistrationno,enterprisetype,majoractivity,activitydescription,Source,Ind_Group,Ind_Monthly
10938,LUGL151520211206212252,HCFAJMSEC00001012127,ECKPR4749P,UDYAM-RJ-01-0023964,Micro,Services,"flour milling, manufacture of food products, m...",tb_udhyam,food,2.a Food Processing
16331,LUGL151520211206212252,HCFAJMSEC00001012127,ECKPR4749P,UDYAM-RJ-01-0023964,Micro,Services,"other semi-processed, processed or instant foo...",tb_udhyam,food,2.a Food Processing
181194,LUGL151520211206212252,HCFAJMSEC00001012127,ECKPR4749P,UDYAM-RJ-01-0023964,Micro,Services,"other semi-processed, processed or instant foo...",tb_udhyam,food,2.a Food Processing
325437,LUGL151520211206212252,HCFAJMSEC00001012127,ECKPR4749P,UDYAM-RJ-01-0023964,Micro,Services,"flour milling, manufacture of food products, m...",tb_udhyam,food,2.a Food Processing


In [257]:
# df_ind_fnl.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df_ind_fnl.xlsx", index=False)
# df_result_blanks.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df_result_blanks.xlsx", index=False)

In [258]:
Prev_rbi_file = pd.read_excel(r"D:\Saurabh\Reports\RBI Reporting\FY 26-27\July'26\NBFC Return base file July'26.xlsx","Sheet1") # reads the first sheet of your excel file
# # df_result_blanks.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df_result_blanks.xlsx", index=False)

In [259]:
columns_to_keep = ['LAN_Number','mst_fnl','ind_fnl']

Prev_rbi_file_1 = Prev_rbi_file[columns_to_keep]

In [260]:
currrent_rbi_file = pd.read_excel(r"D:\Saurabh\Reports\RBI Reporting\FY 26-27\Aug'26\PQR_Aug26_v6_shared.xlsx") # reads the first sheet of your excel file

In [261]:
columns_to_keep = ['LAN_Number','INDUSTRY','SUBINDUSTRY','SCHEME','POS (in Cr)','BRANCH','PAN_NO','LAF','Source','Changed POS (Cr)']

currrent_rbi_file_1 = currrent_rbi_file[columns_to_keep]

In [262]:
currrent_rbi_file_2=pd.merge(currrent_rbi_file_1,Prev_rbi_file_1,on='LAN_Number',how='left')

In [263]:
currrent_rbi_file_2['mst_fnl'].unique()

array(['3.2 Small', '2.2 small', '3.3 Medium', '2.3 Medium', '2.1 micro',
       '3.1 Micro', '2.1 Micro', nan, 'Agriculture', '3.1 micro',
       '2.2 Small'], dtype=object)

In [264]:
currrent_rbi_file_2["mst_fnl"] = currrent_rbi_file_2["mst_fnl"].fillna('NA')
currrent_rbi_file_2["ind_fnl"] = currrent_rbi_file_2["ind_fnl"].fillna('NA')

In [265]:
currrent_rbi_file_2.rename(columns={'PAN_NO': 'pan'}, inplace=True)

In [266]:
currrent_rbi_file_2.rename(columns={'LAF': 'laf'}, inplace=True)

In [267]:
df_ind_fnl.rename(columns={'pan_number': 'pan'}, inplace=True)

In [268]:
df_ind_fnl.rename(columns={'object_pri_key_1': 'laf'}, inplace=True)

In [269]:
df_ind_fnl.columns

Index(['laf', 'agreementno', 'pan', 'udyamregistrationno', 'enterprisetype',
       'majoractivity', 'activitydescription', 'Source', 'Ind_Group',
       'Ind_Monthly'],
      dtype='object')

In [270]:
columns_to_keep = ['laf','agreementno','pan','activitydescription','enterprisetype','majoractivity','Ind_Group','Ind_Monthly']

df_ind_fnl_1 = df_ind_fnl[columns_to_keep]

df_ind_fnl_1.head(2)

,laf,agreementno,pan,activitydescription,enterprisetype,majoractivity,Ind_Group,Ind_Monthly
0,LGFL383520251218082015,None,CHVPR5086D,activities of sound recording in studio or els...,Micro,TRADING,professional services,3.f Professional Services
1,LUGL839620260725113041,None,EIVPG9825E,activities of food service contractors (e.g. f...,Micro,Services,food,2.a Food Processing


In [271]:
currrent_rbi_file_2=pd.merge(currrent_rbi_file_2,df_ind_fnl_1,on='laf',how='left')

In [272]:
# currrent_rbi_file_2['ind_fnl'] = currrent_rbi_file_2['ind_fnl'].isin(['NA']).(currrent_rbi_file_2['Ind_Monthly'])
currrent_rbi_file_2['ind_fnl'] = np.where(currrent_rbi_file_2['ind_fnl'].isin(['NA']), currrent_rbi_file_2['Ind_Monthly'], currrent_rbi_file_2['ind_fnl'])

In [273]:
# filtered_df = currrent_rbi_file_2[currrent_rbi_file_2['LAN_Number'] != 'HCFAHMMLN00001005844']
filtered_df = currrent_rbi_file_2[(currrent_rbi_file_2["LAN_Number"]=="HCFAHMSEC00001045045")] #Filtering dataframe
filtered_df

,LAN_Number,INDUSTRY,SUBINDUSTRY,SCHEME,POS (in Cr),BRANCH,pan_x,laf,Source,Changed POS (Cr),mst_fnl,ind_fnl,agreementno,pan_y,activitydescription,enterprisetype,majoractivity,Ind_Group,Ind_Monthly


In [274]:
currrent_rbi_file_2.loc[
    (currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
           currrent_rbi_file_2['ind_fnl'].str.contains('2') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Micro'),
        'mst_fnl']="2.1 micro"

In [275]:
currrent_rbi_file_2.loc[
   ( currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
          ( currrent_rbi_file_2['ind_fnl'].str.contains('2') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Small')),
        'mst_fnl']="2.2 small"

In [276]:
currrent_rbi_file_2.loc[
    (currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
          ( currrent_rbi_file_2['ind_fnl'].str.contains('2') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Medium')),
        'mst_fnl']="2.3 Medium"

In [277]:
currrent_rbi_file_2.loc[
   ( currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
           (currrent_rbi_file_2['ind_fnl'].str.contains('3') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Micro')),
        'mst_fnl']="3.1 micro"

In [278]:
currrent_rbi_file_2.loc[
   ( currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
          ( currrent_rbi_file_2['ind_fnl'].str.contains('3') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Small')),
        'mst_fnl']="3.2 small"

In [279]:
currrent_rbi_file_2.loc[
   ( currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
          ( currrent_rbi_file_2['ind_fnl'].str.contains('3') &
            currrent_rbi_file_2['enterprisetype'].str.contains('Medium')),
        'mst_fnl']="3.3 Medium"

In [280]:
currrent_rbi_file_2.loc[
   ( currrent_rbi_file_2['mst_fnl'].isin(['NA'])) &
          ( currrent_rbi_file_2['ind_fnl'].str.contains('agriculture') ),
        'mst_fnl']="agriculture"

In [ ]:
# currrent_rbi_file_2=pd.merge(currrent_rbi_file_2,df_ind_fnl_1,on='agreementno',how='left')

In [281]:
# currrent_rbi_file_2['ind_first_letter'] = currrent_rbi_file_2['ind_fnl'].str[0]

# currrent_rbi_file_2['ind_first_letter'] = currrent_rbi_file_2['ind_first_letter'].replace('N', 0)

# currrent_rbi_file_2['ind_first_letter'] = currrent_rbi_file_2['ind_first_letter'].fillna(0)

# currrent_rbi_file_2['ind_first_letter'] = pd.to_numeric(currrent_rbi_file_2['ind_first_letter'])

In [282]:
# currrent_rbi_file_2.loc[
#     currrent_rbi_file_2['mst_fnl'].isin(['NA']) &
#     currrent_rbi_file_2['ind_first_letter']==2 &
#            currrent_rbi_file_2['enterprisetype'].str.contains('Micro'),
#         'mst_fnl']="2.1 micro"

In [283]:
# test=currrent_rbi_file_2.head(2000)
test=currrent_rbi_file_2

In [284]:
# Prev_rbi_file.rename(columns={'agreementno':'lan'},inplace=True)

In [285]:
# test.to_excel(r"C:\Users\saurabh.khopkar\Downloads\test.xlsx", index=False)

In [ ]:
file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
one_day_ago = datetime.now() - timedelta(days=0)
cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
filename="test" + str(cur_time) + ".xlsx"
test.to_excel(file_path + filename, index=False)

In [ ]:
# file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
# one_day_ago = datetime.now() - timedelta(days=1)
# cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
# filename="df_ind_fnl" + str(cur_time) + ".xlsx"
# df_ind_fnl.to_excel(file_path + filename, index=False)

In [238]:
file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
one_day_ago = datetime.now() - timedelta(days=1)
cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
filename="CKYC_as_on_" + str(cur_time) + ".xlsx"
df.to_excel(file_path + filename, index=False)

In [239]:
file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
one_day_ago = datetime.now() - timedelta(days=1)
cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
filename="tb_udhyam_details_As_on_" + str(cur_time) + ".xlsx"
df2.to_excel(file_path + filename, index=False)